# Lab 3: Sieci neuronowe i Deep Learning

## Wstęp

Celem laboratorium jest zapoznanie się z podstawami sieci neuronowych oraz uczeniem głębokim (*deep learning*). Zapoznasz się na nim z następującymi tematami:
- treningiem prostych sieci neuronowych, w szczególności z:
  - regresją liniową w sieciach neuronowych
  - optymalizacją funkcji kosztu
  - algorytmem spadku wzdłuż gradientu
  - siecią typu Multilayer Perceptron (MLP)
- frameworkiem PyTorch, w szczególności z:
  - ładowaniem danych
  - preprocessingiem danych
  - pisaniem pętli treningowej i walidacyjnej
  - walidacją modeli
- architekturą i hiperaprametrami sieci MLP, w szczególności z:
  - warstwami gęstymi (w pełni połączonymi)
  - funkcjami aktywacji
  - regularyzacją: L2, dropout

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from checkers import *

np.set_printoptions(precision=4, suppress=True)

---

Jak już wiemy, model w uczeniu maszynowym to algorytm, który ma w jakimś stopniu modelować rzeczywistość.

Modele parametryczne są funkcjami, których argumentami są wejście $x$ oraz zestaw parametrów $\theta$. Parametrami mogą być:
- współczynniki $\alpha$ i $\beta$ w regresji liniowej
- wagi sieci neuronowej
- wagi $w$ i wyraz wolny $b$ w regresji logistycznej

W literaturze modele z parametrami oznacza się jako funkcje $f(x, \theta)$ lub $f(x; \theta)$.

Dla danych $X$ oraz zestawu parametrów $\theta$ model zwraca predykcje:

$$
\hat y = f(X; \theta)
$$

Dotychczas zarówno do definicji modelu i jak doboru parametrów na dotychczasowych laboratoriach wykorzystywaliśmy wysokopoziomowe interfejsy scikit-learn. Na tym laboratorium, przyjrzymy się w jaki sposób sieci neuronowe oraz inne modele parametryczne mogą zostać zaimplementowane z wykorzystaniem bardziej niskopoziomowego API.

---

W przypadku sieci neuronowych metodą optymalizacji będzie Gradient Descent - spadek wzdłóż gradientu.

Idea jest następująca: definiujemy **funkcję kosztu (cost function)**, nazywaną też funkcją straty (*loss function*),
czyli liczbę $L$ mierzącą, jak bardzo predykcje $\hat y$ różnią się od prawdziwych etykiet $y$. **Uczeniem** parametrów w przypadku tej metody będzie **szukanie parametrów, dla których koszt jest najmniejszy.**

Tak więc, na laboratorium będziemy "uczyć" nasze modele w sposób następujący:

0. **Inicjalizujemy parametry** - zaczynamy od małych, losowych wartości.
1. **Wykonujemy przejście w przód (forward pass)** - liczymy predykcje dla aktualnych parametrów.
2. **Obliczamy funkcję kosztu** - wyznaczamy błąd predykcji $L$.
3. **Wyznaczamy gradient** $\nabla_\theta L$ - wektor pochodnych kosztu po każdym z parametrów; wskazuje kierunek,
   w którym koszt najszybciej **rośnie**.
4. **Aktualizujemy parametry** - robimy mały krok w stronę przeciwną do gradientu, gdzie $\eta$ to **stała ucząca (learning rate)**:

   $$
   \theta \leftarrow \theta - \eta \, \nabla_\theta L
   $$

<img src="figures/petla_uczenia.png" width="800" alt="pętla uczenia">

---
## Część I - Regersja Logistyczna w NumPy

Aby nabyć intuicję dotyczącą tego, w jaki sposób działa propagacja wsteczna i później sieci neuronowe, rozpoczniemy od implementacji regresji logistycznej w bibliotece NumPy - będzie to od nas wymagało zdefiniowania procesu uczącego krok po kroku.

Jako zbiór danych mamy punkty na płaszczyźnie, opisane cechami $x = (x_1, x_2)$, z dwóch klas ($y \in \{0, 1\}$). W naszym przypadku klasy da się
idealnie rozdzielić prostą (są **liniowo separowalne**). 

Do stworzenia modelu przypisującego klasy każdemu z naszych punktów użyjemy **regresji logistycznej** (laboratorium 2):
$$
\hat p = \sigma(w^\top x + b) = \frac{1}{1 + e^{-(w^\top x + b)}}
$$
gdzie $\hat p$ to prawdopodobieństwo klasy 1, a $z = w^\top x + b$ to **logit**. **Granica decyzyjna
(decision boundary)** to prosta 

$$ 
w^\top x + b = 0 
$$ 

czyli 
$$
\hat p = 0.5
$$

Proces uczenia będzie obracał i przesuwał tę prostą, aż oddzieli klasy.

<img src="figures/granica_decyzyjna.png" width="760" alt="granica decyzyjna">

Skupimy się tutaj na rozpisaniu każdego kroku z pętli uczącej używając biblioteki `Numpy`.

Na początku zobaczmy jak wygląda nasz dataset. Mały opis funkcji pomocniczych:
- `plot_decision_boundary` pyta model o $\hat p$ w każdym punkcie gęstej siatki: tło pokazuje prawdopodobieństwo
  (niebieski - klasa 0, czerwony - klasa 1), a czarna linia - granicę $\hat p = 0.5$;
- `plot_training` pokazuje wynik treningu: krzywą uczenia (koszt w kolejnych epokach, czyli `history` zwracane przez
  pętlę uczącą) oraz granicę decyzyjną wytrenowanego modelu.

Model przekazujemy jako funkcję `predict_proba`, która dla tablicy punktów $(M, 2)$ zwraca $M$ prawdopodobieństw -
dzięki temu te same funkcje zadziałają dla modeli w NumPy i w PyTorchu.

In [ ]:
def make_linear_dataset(n=200, seed=0):
    # points from [-3, 3]^2 labelled by the side of the line 1.5 x1 - x2 + 0.5 = 0,
    # points too close to the line are dropped, so the classes are clearly separated
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3, 3, size=(3 * n, 2))
    z = X @ np.array([1.5, -1.0]) + 0.5
    keep = np.abs(z) > 0.6
    X, z = X[keep][:n], z[keep][:n]
    return X, (z > 0).astype(float)


def plot_dataset(X, y, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.3)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    if title:
        ax.set_title(title)
    return ax


def plot_decision_boundary(predict_proba, X, y, ax=None, title=None, margin=0.5, resolution=200):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - margin, X[:, 0].max() + margin, resolution),
        np.linspace(X[:, 1].min() - margin, X[:, 1].max() + margin, resolution),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]  # (resolution^2, 2) - each row is one grid point
    probs = np.asarray(predict_proba(grid)).reshape(xx.shape)
    ax.contourf(xx, yy, probs, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.4)  # soft boundary
    if probs.min() < 0.5 < probs.max():
        ax.contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=2)  # hard boundary at 0.5
    plot_dataset(X, y, ax=ax, title=title)
    return ax


def plot_training(history, predict_proba, X, y, title="", compare=None, mark_errors=False):
    # learning curve of one training run + decision boundary of the trained model
    # compare: optional {label: history} of other runs drawn on the same learning curve plot
    wrong = (np.asarray(predict_proba(X)) >= 0.5) != y
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.2, 1]})
    for label, other in (compare or {}).items():
        ax1.plot(other, label=label, alpha=0.7)
    ax1.plot(history, label=title or "trening", color="black" if compare else None)
    ax1.set_xlabel("epoka")
    ax1.set_ylabel("koszt")
    ax1.set_title(f"Krzywa uczenia: {history[0]:.3f} → {history[-1]:.3f}")
    ax1.grid(alpha=0.3)
    if compare:
        ax1.legend()
    plot_decision_boundary(predict_proba, X, y, ax=ax2, title=f"{title} - dokładność {1 - wrong.mean():.1%}")
    if mark_errors:
        ax2.scatter(X[wrong, 0], X[wrong, 1], marker="x", color="black", s=20, label="błędne predykcje")
        ax2.legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    plt.show()




In [ ]:
X_lin, y_lin = make_linear_dataset()
print("X_lin:", X_lin.shape, " y_lin:", y_lin.shape)

ax = plot_dataset(X_lin, y_lin, title="Dane liniowo separowalne")
grid_x1 = np.linspace(-3, 3, 10)
ax.plot(grid_x1, 1.5 * grid_x1 + 0.5, "--", color="gray", label="prawdziwa granica\n(nieznana modelowi)")
ax.set_ylim(-3.3, 3.3)
ax.legend(loc="lower right", fontsize=8)
plt.show()

### Zadanie 1: Definiowanie pochodnej

[Pochodna](https://en.wikipedia.org/wiki/Derivative) $f'(x)$ to nachylenie stycznej do wykresu funkcji w punkcie $x$
([animacja: Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Tangent_function_animation.gif)):

<img src="https://upload.wikimedia.org/wikipedia/commons/2/2d/Tangent_function_animation.gif" width="300" alt="pochodna jako nachylenie stycznej">

Można ją policzyć **analitycznie** (na kartce, z reguł różniczkowania) zgodnie z definicją:

$$
f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}
$$

albo
[**numerycznie**](https://en.wikipedia.org/wiki/Numerical_differentiation) - przesuwając $x$ o małe $h$ i mierząc
zmianę wartości funkcji. Użyjemy **różnicy centralnej (central difference)**:
$$
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}
$$

Uzupełnij:
- `analytical_df(x)` - pochodna funkcji $f(x) = 3x^2 - 4x + 5$ wyprowadzona na kartce,
- `df(func, x, h)` - pochodna numeryczna **dowolnej** funkcji `func` w punkcie `x`.

Obie funkcje mają działać także dla tablic NumPy (element po elemencie).

In [ ]:
def f(x):
    # helper function (given)
    return 3 * x**2 - 4 * x + 5

In [ ]:
def analytical_df(x):
    # derivative of f(x) = 3x^2 - 4x + 5, derived on paper
    # implement me!
    raise NotImplementedError


def df(func, x, h=1e-5):
    # numerical derivative of func at x (central difference); works elementwise for numpy arrays
    # implement me!
    raise NotImplementedError

In [ ]:
sprawdz_zadanie_1(analytical_df, df)

Porównajmy pochodne analityczne (linia) i numeryczne (punkty) dla $f$ oraz dla funkcji z animacji:

$$
g(x) = x \sin(x^2) + 1
$$

której pochodna to:

$$
g'(x) = \sin(x^2) + 2x^2 \cos(x^2)
$$

In [ ]:
def g(x):
    return x * np.sin(x**2) + 1


def analytical_dg(x):
    return np.sin(x**2) + 2 * x**2 * np.cos(x**2)


fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, func, dfunc, name, xs in (
    (axes[0], f, analytical_df, "f(x) = 3x² − 4x + 5", np.linspace(-1.5, 3, 200)),
    (axes[1], g, analytical_dg, "g(x) = x·sin(x²) + 1", np.linspace(-2, 3, 400)),
):
    ax.plot(xs, func(xs), color="gray", label="funkcja")
    ax.plot(xs, dfunc(xs), lw=2, label="pochodna analityczna")
    ax.plot(xs[::12], df(func, xs[::12]), "o", ms=4, label="pochodna numeryczna (df)")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(name)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

Pochodna numeryczna wymaga tylko możliwości policzenia funkcji, ale dla modelu z $P$ parametrami gradient kosztuje
$2P$ obliczeń kosztu. Dlatego w praktyce, tam gdzie to możliwe, używa się pochodnych analitycznych. Pochodne numeryczne służą do
ich **sprawdzania** (*gradient checking*) - tak działają sprawdzarki w tym notatniku.

### Zadanie 2: Definicja modelu i forward pass


Zdefiniujemy teraz nasz model jako klasę `LogisticRegression`. Obiekt tej klasy przechowuje parametry modelu
(`self.w`, `self.b`) i potrafi policzyć predykcje, czyli wykonać **forward pass** (metoda `forward`).
Z bardzo podobnymi klasami spotkamy się później w PyTorchu - tam również model to klasa z parametrami
i metodą `forward`.

Przypomnijmy wzory. Dla punktu $x$ model liczy najpierw logit, czyli kombinację liniową cech:

$$
z = w^\top x + b = w_1 x_1 + w_2 x_2 + b
$$

a potem przepuszcza go przez funkcję sigmoidalną:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Wynikiem jest prawdopodobieństwo klasy 1:

$$
\hat p = \sigma(z)
$$

Dla wszystkich $N$ punktów naraz (wiersze macierzy $X$) to samo zapisujemy wektorowo:

$$
z = X w + b
$$

Uzupełnij:
- `sigmoid(z)` - element po elemencie,
- `LogisticRegression.__init__` - `self.w`: wektor `(n_features,)` z rozkładu normalnego o odchyleniu 0.1
  (`rng.normal`), `self.b`: `0.0`,
- `LogisticRegression.forward` - prawdopodobieństwa klasy 1 dla wszystkich punktów `X` naraz, kształt `(N,)`
  (`X @ w + b` ma kształt $(N, 2) \cdot (2,) \to (N,)$, a `np.exp` działa element po elemencie).

In [ ]:
def sigmoid(z):
    # implement me! - delete the `raise` line and replace every `...` below
    raise NotImplementedError
    return ...  # 1 / (1 + e^(-z)), elementwise (np.exp)


class LogisticRegression:
    def __init__(self, n_features, seed=0):
        rng = np.random.default_rng(seed)
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        self.w = ...  # weights, shape (n_features,): normal distribution, mean 0, std 0.1 (rng.normal)
        self.b = ...  # bias, a single number: 0.0

    def forward(self, X):
        # X: (N, n_features) -> probabilities of class 1, shape (N,)
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        z = ...       # 1. logits: X @ w + b                    (N,)
        return ...    # 2. activation: sigmoid of the logits    (N,)

    def predict(self, X, threshold=0.5):
        # hard 0/1 predictions (given)
        return (self.forward(X) >= threshold).astype(float)

In [ ]:
sprawdz_zadanie_2(sigmoid, LogisticRegression)

Zwizualizujmy teraz, jaką granicę decyzyjną wyznacza nasz model z losowo zainicjalizowanymi parametrami.

In [ ]:
model = LogisticRegression(2, seed=1)
print("parametry:  w =", model.w, "  b =", model.b)


acc = np.mean(model.predict(X_lin) == y_lin)
plot_decision_boundary(model.forward, X_lin, y_lin, title=f"Losowe parametry - dokładność {acc:.0%}")
plt.show()

### Zadanie 3: Funkcja kosztu

Jako funkcji kosztu użyjemy **binarnej entropii krzyżowej (binary cross-entropy, BCE)**, zdefiniowanej w laboratorium 2:
$$
L = BCE(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]
$$

Koszt pojedynczego punktu, czyli wyraz sumy wraz z minusem, będziemy dalej oznaczać $\ell_i$. Cały graf obliczeń -
od danych i parametrów do kosztu:

<img src="figures/regresja_graf.png" width="880" alt="graf obliczeń">

Uzupełnij `bce` - średni koszt wszystkich punktów (jedna liczba).
`np.clip` w szablonie chroni przed `log(0)`.

In [ ]:
def bce(y, p_hat, eps=1e-12):
    # mean binary cross-entropy - a single number
    p_hat = np.clip(p_hat, eps, 1 - eps)  # protects against log(0)
    # implement me!
    raise NotImplementedError

In [ ]:
sprawdz_zadanie_3(bce)

Na wykresach poniżej widać koszt punktu oraz jego pochodną po logicie $z$ (liczoną Twoją funkcją `df`)
w porównaniu z MSE.
Gdy model bardzo się myli, pochodna BCE wynosi około $-1$ (silny sygnał do poprawy), a pochodna MSE jest prawie
zerowa - dlatego do klasyfikacji używamy entropii krzyżowej.

In [ ]:
p_grid = np.linspace(0.001, 0.999, 300)
z_grid = np.linspace(-8, 8, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(p_grid, -np.log(p_grid), label="y = 1:  −log(p̂)")
ax1.plot(p_grid, -np.log(1 - p_grid), label="y = 0:  −log(1 − p̂)")
ax1.set_xlabel("predykcja p̂")
ax1.set_title("koszt punktu")
ax1.set_ylim(0, 5)
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

ax2.plot(z_grid, df(lambda z: -np.log(sigmoid(z)), z_grid), label="BCE")
ax2.plot(z_grid, df(lambda z: (sigmoid(z) - 1.0) ** 2, z_grid), label="MSE")
ax2.axvspan(-8, 0, color="gray", alpha=0.1)
ax2.text(-4, -0.85, "model się myli", ha="center", fontsize=9)
ax2.set_xlabel("logit z")
ax2.set_title("pochodna kosztu po z (punkt z klasy 1)")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"koszt losowego modelu: {bce(y_lin, model.forward(X_lin)):.4f}  (dla p̂ = 0.5 wszędzie: ln 2 = {np.log(2):.4f})")

### Zadanie 4: Gradient i propagacja wsteczna

Pełniejsze wyjaśnienie (definicje, dlaczego gradient wskazuje kierunek najszybszego wzrostu, dobór stałej uczącej)
znajdziesz w pliku [notatki_gradient_backprop.md](notatki_gradient_backprop.md).

**Gradient** to wektor pochodnych cząstkowych kosztu po każdym z parametrów - u nas
$\nabla_\theta L = \left[ \frac{\partial L}{\partial w_1}, \frac{\partial L}{\partial w_2}, \frac{\partial L}{\partial b} \right]$.
Wskazuje kierunek, w którym koszt najszybciej **rośnie**, więc parametry przesuwamy w stronę przeciwną,
o krok skalowany stałą uczącą $\eta$ (Zadanie 5):

$$
\theta \leftarrow \theta - \eta \, \nabla_\theta L
$$

<!-- TODO: zastąpić gifem -->
<img src="figures/spadek_gradientu_2d.png" width="880" alt="spadek wzdłuż gradientu w 2D">

Koszt to złożenie prostych funkcji (graf z Zadania 3): $z_i = w^\top x_i + b$, $\hat p_i = \sigma(z_i)$,
$\ell_i(y_i, \hat p_i)$. Z **reguły łańcuchowej (chain rule)** pochodna po parametrze to iloczyn pochodnych lokalnych
kolejnych ogniw:

$$
\frac{\partial \ell_i}{\partial w_j} =
\underbrace{\frac{\partial \ell_i}{\partial \hat p_i}}_{\text{BCE}} \cdot
\underbrace{\frac{\partial \hat p_i}{\partial z_i}}_{\text{sigmoida}} \cdot
\underbrace{\frac{\partial z_i}{\partial w_j}}_{x_{ij}}
$$

**Propagacja wsteczna** liczy te iloczyny od końca grafu, dzięki czemu wspólną część łańcucha liczymy raz dla wszystkich
parametrów. Koszt batcha (u nas całego zbioru treningowego) to średnia $L = \frac{1}{N} \sum_{i=1}^{N} \ell_i$, a każdy
parametr wpływa na niego przez wszystkie punkty. Najpierw liczymy więc dla każdego punktu, jak koszt zależy od jego logitu:

$$
\delta_i = \frac{\partial L}{\partial z_i} = \frac{1}{N} \cdot \frac{\partial \ell_i}{\partial \hat p_i} \cdot \frac{\partial \hat p_i}{\partial z_i}
$$

a potem dokładamy ostatnie ogniwo ($\frac{\partial z_i}{\partial w_j} = x_{ij}$, $\frac{\partial z_i}{\partial b} = 1$)
i sumujemy wkłady punktów:

$$
\frac{\partial L}{\partial w_j} = \sum_{i=1}^{N} \delta_i \, x_{ij}
\qquad
\frac{\partial L}{\partial b} = \sum_{i=1}^{N} \delta_i
$$

Uzupełnij `backward` - funkcja tylko zwraca gradient `(grad_w, grad_b)`, a parametry zmienimy w pętli uczącej
(Zadanie 5). Gradient ma zawsze kształt parametru: `grad_w` - `(2,)`, `grad_b` - liczba. Kroki w szablonie to kolejne
ogniwa łańcucha:

1. `dloss_dp` - pochodna BCE, podana analitycznie:
   $\dfrac{\partial \ell_i}{\partial \hat p_i} = \dfrac{\hat p_i - y_i}{\hat p_i (1 - \hat p_i)}$
   (`df(lambda p: bce(y, p), p_hat)` nie zadziała, bo `bce` zwraca średnią, a nie koszt każdego punktu),
2. `dp_dz` - pochodna sigmoidy, policzona numerycznie funkcją `df` z Zadania 1,
3. `delta` - wszystkie $\delta_i$ naraz, jako wektor o kształcie `(N,)` (bez pętli po punktach),
4. `grad_w` - sumę po punktach dla wszystkich wag naraz daje `X.T @ delta`,
5. `grad_b` - suma `delta`.

Sprawdzarka porówna wynik z gradientem policzonym „na siłę” (przesuwanie parametrów o $\pm h$) i ze wzorem
analitycznym, który wychodzi po policzeniu obu pochodnych lokalnych na kartce:

$$
\frac{\partial L}{\partial w} = \frac{1}{N} X^\top (\hat p - y)
$$

Przykład ręcznie napisanego przejścia w przód i wstecz w NumPy:
[Learning PyTorch with Examples - Warm-up: numpy](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#warm-up-numpy).

In [ ]:
def backward(model, X, y):
    # gradient of the mean BCE cost w.r.t. model parameters -> (grad_w, grad_b)
    # numpy example of a manual backward pass: https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#warm-up-numpy
    N = len(y)
    # forward pass, keeping intermediate values (given)
    z = X @ model.w + model.b  # (N,)
    p_hat = sigmoid(z)         # (N,)

    # implement me! - delete the `raise` line and replace every `...` below
    raise NotImplementedError
    dloss_dp = ...  # 1. local derivative of each point's cost w.r.t. p_hat, analytically (formula above)  (N,)
    dp_dz = ...     # 2. local derivative of sigmoid w.r.t. z, numerically with df                     (N,)
    delta = ...     # 3. chain rule: dloss_dp * dp_dz, divided by N (L is the mean over the batch)     (N,)
    grad_w = ...    # 4. sum over the batch of delta_i * x_ij, for all weights at once (X.T @ ...)     (2,)
    grad_b = ...    # 5. sum over the batch of delta_i (because dz/db = 1)                              number
    return grad_w, grad_b

In [ ]:
sprawdz_zadanie_4(backward, LogisticRegression)

### Zadanie 5: Pętla ucząca

Mając już wszystkie potrzebne elementy, możemy teraz zaimplementować **pętlę uczącą (training loop)**, czyli iteracyjne optymalizowanie parametrów korzystające, w naszym przypadku, z Gradient Descent.

Zmieniaj parametry w miejscu (przypisz nowe wartości do `model.w` i `model.b`), a do historii dopisuj koszt policzony **przed** aktualizacją, jako liczbę (`float(...)`).
Po sprawdzeniu wytrenujemy model, a `plot_training` narysuje krzywą uczenia z Twojej historii i granicę decyzyjną.
Podobną pętlę w NumPy znajdziesz w tutorialu [Learning PyTorch with Examples - Warm-up: numpy](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#warm-up-numpy).

In [ ]:
def train(model, X, y, lr=0.5, epochs=100):
    # full-batch gradient descent; modifies the model in place, returns cost history (one value per epoch)
    # numpy example of a training loop: https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#warm-up-numpy
    history = []
    for epoch in range(epochs):
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        p_hat = ...           # 1. forward pass: probabilities for all points (model.forward)    (N,)
        loss = ...            # 2. cost before the update (bce)                                   number
        history.append(float(loss))
        grad_w, grad_b = ...  # 3. gradient of the cost (backward)
        model.w = ...         # 4. update: step against the gradient, scaled by lr
        model.b = ...         #    the same for the bias
    return history

In [ ]:
sprawdz_zadanie_5(train, LogisticRegression)

In [ ]:
model = LogisticRegression(2, seed=1)
history = train(model, X_lin, y_lin, lr=0.5, epochs=100)
print(f"wyuczone parametry: w = {model.w}, b = {model.b:.4f}")
plot_training(history, model.forward, X_lin, y_lin, title="NumPy")

Wypróbuj inne wartości `lr` (np. `0.01`, `5`, `50`) i liczby epok - jak zmienia się krzywa uczenia?

---
## Część II - PyTorch

Dotychczas do tworzenia pętli uczącej używaliśmy matematycznych definicji napisanych w Numpy. Mimo wszystko, większość tworzenia i szkolenia modeli jest raczej schematyczna, więc do tego typu operacji zwykle będziemy chcieli używać frameworka. Jednym z nich jest PyTorch - udostępniony przez zespół Facebook AI Research (FAIR), jego publiczna wersja pojawiła się w 2017 roku. Powstał na bazie wcześniejszych prac nad narzędziami do obliczeń tensorowych i uczenia głębokiego.

PyTorch to w gruncie rzeczy NumPy + GPU + [automatyczne różniczkowanie (Autograd)](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html). Właśnie automatyczne różniczkowanie pozwoli nam ograniczyć pisanie kodu związanego z poprawną implementacją gradient descent.

Najważniejsze moduły:
- `torch` - tensory i podstawowe funkcje, np. `Tensor`, `from_numpy()`,
- `torch.nn` - klocki sieci neuronowych, np. `Linear`, `Sigmoid`, `BCELoss`,
- `torch.optim` - optymalizatory, czyli warianty spadku wzdłuż gradientu.

Pętla ucząca ma te same kroki co w Części I:

<img src="figures/petla_uczenia_pytorch.png" width="800" alt="pętla w PyTorchu">

| Krok | Część I (NumPy) | PyTorch |
|---|---|---|
| parametry + przejście w przód | `LogisticRegression.forward` | `nn.Module`, `nn.Linear`, `nn.Sigmoid`; wywołanie `model(X)` |
| funkcja kosztu | `bce` | `nn.BCELoss()` |
| gradient (backward pass) | `backward` | `loss.backward()` |
| aktualizacja (optymalizator) | `w - lr * grad_w` | `optimizer.step()` (+ `optimizer.zero_grad()`) |

### Tensory i autograd

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

API tensorów jest bardzo podobne do NumPy, ale na dwie rzeczy trzeba uważać:

- **Typ liczb** - NumPy domyślnie tworzy liczby typu `float64`, a warstwy w PyTorchu pracują na `float32`.
  Jeśli podasz modelowi tensor `float64`, dostaniesz błąd o niezgodnych typach. Dlatego dane z NumPy zamieniamy
  metodą `.float()` albo od razu tworzymy je przez `torch.tensor(..., dtype=torch.float32)`.

- **Output `shape`** - model zwraca jedną kolumnę wyników, czyli tensor o kształcie $(N, 1)$, a w Numpy etykiety mamy zwykle
  jako płaski wektor o kształcie $(N,)$. Funkcja kosztu wymaga, żeby predykcje i etykiety miały **ten sam** kształt,
  więc zamieniamy wektor etykiet w kolumnę metodą `.unsqueeze(1)` (dokłada wymiar o rozmiarze 1).
  W drugą stronę, z kolumny do płaskiego wektora, działa `.squeeze(1)`.

In [ ]:
a = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.from_numpy(a)  # shares memory with the numpy array
print("tensor z NumPy:", t.dtype, tuple(t.shape))

t32 = t.float()
print("po .float():", t32.dtype)

result = t32 @ torch.ones(2)  # matrix multiplication, as in NumPy
print("t32 @ ones:", result)

labels = torch.tensor([1.0, 0.0, 1.0])
labels_col = labels.unsqueeze(1)
print("etykiety:", tuple(labels.shape))
print("po unsqueeze(1):", tuple(labels_col.shape))

### Autograd - jak PyTorch liczy gradienty

W Części I gradient liczyliśmy sami: w przejściu w przód zapamiętywaliśmy wartości pośrednie (`z`, `p_hat`), a w funkcji
`backward` mnożyliśmy lokalne pochodne zgodnie z regułą łańcuchową. **Autograd** (automatyczne różniczkowanie) robi
dokładnie to samo, tylko automatycznie. Działa w czterech krokach:

1. **`requires_grad=True`** - oznaczamy tensory, po których chcemy liczyć pochodne. W modelach robi to za nas
   `nn.Module`: wszystkie parametry (wagi, biasy) mają tę flagę ustawioną od razu. Dane wejściowe i etykiety jej nie
   potrzebują, bo ich nie uczymy.


2. **Graf obliczeń** - każda operacja na oznaczonym tensorze (`*`, `+`, `**`, `@`, `sigmoid`, ...) tworzy nowy tensor,
   który pamięta, z czego powstał i jak policzyć swoją lokalną pochodną. Tę informację widać w atrybucie `grad_fn`.
   Graf powstaje „w locie”, podczas zwykłego wykonywania kodu.


3. **`.backward()`** - wywołujemy na wyniku, który jest jedną liczbą (zwykle na koszcie). PyTorch wykonuje backward pass:
   przechodzi graf od końca i mnoży lokalne pochodne (propagacja wsteczna), tak jak nasza funkcja `backward`. Pochodne są liczone ze wzorów analitycznych,
   więc są dokładne, a nie przybliżone jak w `df`.


4. **`.grad`** - wynik trafia do atrybutu `.grad` każdego tensora z `requires_grad=True`.

Zachęcam do zapoznania się z tym, jak sam Autograd funkcjonuje i co dzieje się pod spodem: [A Gentle Introduction to torch.autograd](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html), [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html), [automatyczne różniczkowanie na Wikipedii](https://en.wikipedia.org/wiki/Automatic_differentiation) oraz kurs [micrograd](https://www.youtube.com/watch?v=VMj-3S1tku0) Andreja Karpathy'ego, który przedstawia uproszczoną implementację tego narzędzia.


Sprawdźmy, jak Autograd działa na funkcji $f(x) = 3x^2 - 4x + 5$ z Zadania 1:

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = 3 * x**2 - 4 * x + 5
print("y:", y)
print("y.grad_fn:", y.grad_fn)  # y remembers the last operation that created it
print("x.grad przed backward():", x.grad)

y.backward()
print("x.grad po backward():", x.grad.item())
print("analytical_df(3):    ", analytical_df(3.0))
print("df(f, 3):            ", df(f, 3.0))

Ważne: PyTorch nie nadpisuje gradientu, tylko **dodaje** nowy do tego, który już jest w `.grad` (akumuluje go).

Bywa to przydatne, ale w zwykłej pętli uczącej oznacza, że przed każdym `backward()` **trzeba gradienty wyzerować** -
inaczej w każdej epoce byłyby coraz większe. Metody kończące się na `_`, jak `.zero_()`, zmieniają tensor w miejscu.
W pętli uczącej zerujemy wszystkie parametry naraz przez `optimizer.zero_grad()`.

In [ ]:
y = 3 * x**2 - 4 * x + 5
y.backward()
print("po drugim backward():", x.grad.item(), "<- gradienty się sumują!")

x.grad.zero_()
print("po x.grad.zero_():", x.grad.item())

Śledzenie operacji w autogradzie kosztuje czas i pamięć, a nie zawsze jest potrzebne - np. przy liczeniu predykcji albo rysowaniu
wykresów. Są na to dwa sposoby:

- blok `with torch.no_grad():` - operacje wewnątrz nie są zapisywane w grafie,

- `.detach()` - zwraca ten sam tensor, ale „odcięty” od grafu; przydaje się np. przed zamianą na NumPy
  (`tensor.detach().numpy()`).

Jednoelementowy tensor zamieniamy na zwykłą liczbę Pythona metodą `.item()`.

In [ ]:
with torch.no_grad():
    y = 3 * x**2 - 4 * x + 5
print("w torch.no_grad():  requires_grad =", y.requires_grad, " grad_fn =", y.grad_fn)

x_detached = x.detach()
print("po x.detach():      requires_grad =", x_detached.requires_grad)
print("y.item():", y.item(), type(y.item()))

### Modele w PyTorchu: `nn.Module`

Modele w PyTorchu to klasy dziedziczące po `nn.Module` - podobnie jak nasza klasa `LogisticRegression` z Zadania 2.
Po kolei, co w takiej klasie robią poszczególne metody:

| Metoda | Kto ją definiuje | Co robi |
|---|---|---|
| `__init__` | my | tworzy warstwy i zapisuje je jako atrybuty (`self.linear = nn.Linear(...)`). Zawsze zaczynamy od `super().__init__()`. Każdy atrybut, który jest `nn.Module` (np. warstwa) albo `nn.Parameter`, zostaje zarejestrowany, a jego parametry dostają `requires_grad=True`. |
| `forward(X)` | my | przejście w przód: z danych `X` liczy wyjście modelu. To jedyna metoda, którą musimy napisać. |
| `model(X)` | PyTorch | wywołuje `forward(X)` (plus mechanizmy wewnętrzne PyTorcha), dlatego nigdy nie wołamy `forward` wprost. |
| backward pass | autograd | **nie definiujemy** metody `backward` - autograd buduje ją sam z operacji wykonanych w `forward` i działa na tensorach. Wywołujemy `loss.backward()` na koszcie. |
| `parameters()` | PyTorch | zwraca wszystkie zarejestrowane parametry modelu; przekazujemy je optymalizatorowi. |
| `zero_grad()` | PyTorch | zeruje gradienty wszystkich parametrów (to samo robi `optimizer.zero_grad()`). |
| `train()`, `eval()` | PyTorch | przełączają tryb treningowy i ewaluacyjny (niektóre warstwy działają inaczej podczas treningu). |

Podstawowym blokiem operacji w Torchu jest [`nn.Linear(in_features, out_features)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) - to nasze `X @ w + b`. Zobaczmy, jakie parametry rejestruje:

In [ ]:
layer = nn.Linear(2, 1)
for name, param in layer.named_parameters():
    print(name, tuple(param.shape), "requires_grad =", param.requires_grad)

print("wyjście dla 3 punktów:", tuple(layer(torch.ones(3, 2)).shape))

### Zadanie 6: Regresja logistyczna w PyTorchu

Napisz regresję logistyczną jako `nn.Module`: warstwa liniowa z `n_features` wejściami i jednym wyjściem (logitem),
a po niej sigmoida (`nn.Sigmoid`). Zwróć uwagę, że `nn.Linear` przechowuje wagi jako macierz
`(out_features, in_features)`, czyli `(1, 2)` dla naszych danych. Przykład podobnej klasy: [Build the Neural Network - Define the Class](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#define-the-class),
a także [Learning PyTorch with Examples - Custom nn Modules](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#pytorch-custom-nn-modules).

Uzupełnij `LogisticRegressionTorch`. Użyj dokładnie nazw atrybutów `self.linear` i `self.activation`.

In [ ]:
class LogisticRegressionTorch(nn.Module):
    # example of a similar class: https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#define-the-class
    def __init__(self, n_features):
        super().__init__()
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        self.linear = ...      # nn.Linear: n_features inputs -> 1 output (logit)
        self.activation = ...  # nn.Sigmoid

    def forward(self, X):
        # X: (N, n_features) -> probabilities of class 1, shape (N, 1)
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        logits = ...  # 1. linear layer applied to X            (N, 1)
        return ...    # 2. activation applied to the logits     (N, 1)

In [ ]:
sprawdz_zadanie_6(LogisticRegressionTorch)

Sprawdźmy, że model w PyTorchu liczy te same gradienty co nasza implementacja w NumPy. Kopiujemy parametry z modelu NumPy do modelu PyTorcha i porównujemy
gradient z `loss.backward()` z Twoją funkcją `backward`.

In [ ]:
X_lin_t = torch.tensor(X_lin, dtype=torch.float32)               # (N, 2)
y_lin_t = torch.tensor(y_lin, dtype=torch.float32).unsqueeze(1)  # (N, 1)

np_model = LogisticRegression(2, seed=1)
torch_model = LogisticRegressionTorch(2)
with torch.no_grad():  # copying parameters is not part of the computation graph
    torch_model.linear.weight.copy_(torch.tensor(np_model.w).view(1, 2))
    torch_model.linear.bias.fill_(np_model.b)

loss = nn.BCELoss()(torch_model(X_lin_t), y_lin_t)
loss.backward()
grad_w, grad_b = backward(np_model, X_lin, y_lin)

print(f"koszt:  NumPy {bce(y_lin, np_model.forward(X_lin)):.6f}   PyTorch {loss.item():.6f}")
print(f"dL/dw:  NumPy {grad_w}   PyTorch {torch_model.linear.weight.grad.numpy().ravel()}")
print(f"dL/db:  NumPy {grad_b:.6f}   PyTorch {torch_model.linear.bias.grad.item():.6f}")

### Optymalizatory

**Optymalizator** wykonuje krok aktualizacji parametrów - krok 4 pętli uczącej. W Części I robiliśmy to ręcznie z użyciem Gradient Descent:

$$
\theta \leftarrow \theta - \eta \, \nabla_\theta L
$$

W PyTorchu optymalizator dostaje przy tworzeniu listę parametrów modelu (`model.parameters()`) i stałą uczącą, a potem:

- `optimizer.zero_grad()` zeruje gradienty wszystkich tych parametrów,

- `optimizer.step()` aktualizuje parametry na podstawie gradientów, które `loss.backward()` zapisał w `.grad`.

**Ile danych na jeden krok?** Dotychczas liczyliśmy gradient na **całym** zbiorze danych, więc jeden krok to jedna
epoka. To jest **spadek wzdłuż gradientu (gradient descent, GD)**, nazywany też *full batch GD*. Dla dużych zbiorów
jest to często niewykonalne - cały zbiór nie mieści się w pamięci, a jeden krok trwa bardzo długo. Dlatego wymyślono:
- **stochastyczny spadek wzdłuż gradientu (stochastic gradient descent, SGD)** - gradient liczymy na jednym
  losowym przykładzie i od razu aktualizujemy parametry. To dość grube przybliżenie gradientu, ale pozwala robić
  szybko dużo małych kroków;
- **minibatch gradient descent** - kompromis używany w praktyce: gradient liczymy na losowej **paczce (batch)**
  $B$ przykładów, np. 32, 64 czy 128:

$$
L_B = \frac{1}{|B|} \sum_{i \in B} \ell_i
$$

$$
\theta \leftarrow \theta - \eta \, \nabla_\theta L_B
$$

Jedna epoka to wtedy wiele kroków - po jednym na każdą paczkę. Rzadko wspominanym, a ważnym faktem jest to, że
losowość paczek sama w sobie działa jak [regularyzacja](https://arxiv.org/abs/2101.12176), więc wielkość paczki
(`batch_size`) to także hiperparametr. W PyTorchu paczki tworzy się klasą `DataLoader` - poznasz ją w dalszej części
laboratorium. Uwaga na nazewnictwo: `optim.SGD` po prostu wykonuje krok w przeciwną stronę niż gradient, który mu
policzymy - jeśli liczymy go na całym zbiorze (tak jak tutaj), to jest to zwykły GD.

**Sprytniejsze kroki.** Zwykły krok zależy tylko od bieżącego gradientu. Popularne ulepszenia:
- **momentum** (`optim.SGD(..., momentum=0.9)`) - krok uwzględnia też kierunek poprzednich kroków, jak kula
  tocząca się z górki. Pomaga przejść przez płaskie fragmenty funkcji kosztu i tłumi oscylacje;
- **[Adam](https://arxiv.org/abs/1412.6980)** (`optim.Adam`) - łączy momentum z osobną, automatycznie dobieraną
  wielkością kroku dla każdego parametru. Uczy szybko i daje dobre wyniki nawet przy niekoniecznie idealnie dobranej
  stałej uczącej. W PyTorchu najlepiej korzystać z jego wersji `AdamW`, która poprawnie łączy go z regularyzacją L2.
  Jest to zasadniczo domyślny wybór przy treningu współczesnych sieci neuronowych.

**Zmiana stałej uczącej w trakcie treningu.** Im większa stała ucząca, tym większe kroki - uczymy szybciej, ale
możemy „przeskakiwać” minimum. Często zaczyna się więc od większej stałej uczącej i zmniejsza ją w trakcie treningu.
Nazywa się to **learning rate scheduling**; w PyTorchu służą do tego klasy z `torch.optim.lr_scheduler`.

| Optymalizator | PyTorch | Typowa stała ucząca |
|---|---|---|
| (minibatch) SGD | `optim.SGD(model.parameters(), lr=0.1)` | $10^{-2}$ - $10^{-1}$ |
| SGD z momentum | `optim.SGD(model.parameters(), lr=0.1, momentum=0.9)` | $10^{-2}$ - $10^{-1}$ |
| Adam / AdamW | `optim.AdamW(model.parameters(), lr=1e-3)` | $10^{-4}$ - $10^{-2}$ |

### Zadanie 7: Pętla ucząca w PyTorchu

Napisz pętlę uczącą w PyTorchu z optymalizatorem przekazanym jako argument. Do historii zapisuj
`loss.item()` (zwykła liczba) - sam tensor `loss` trzymałby w pamięci cały graf obliczeń.
Przykład podobnej pętli: [Optimizing Model Parameters - Full Implementation](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html#full-implementation)
oraz [Learning PyTorch with Examples - PyTorch: optim](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#pytorch-optim);
dokumentacja: [`nn.BCELoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html),
[`torch.optim`](https://docs.pytorch.org/docs/stable/optim.html).

Uzupełnij `train_torch`. Optymalizator jest argumentem, więc tej samej funkcji użyjemy później z innym optymalizatorem.

In [ ]:
def train_torch(model, optimizer, X, y, epochs=100):
    # X: (N, d), y: (N, 1), both float32; returns cost history (floats, one per epoch, before the update)
    # example of a similar loop: https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html#full-implementation
    loss_fn = nn.BCELoss()
    history = []
    for epoch in range(epochs):
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        ...                  # 1. zero the gradients of all parameters (optimizer)
        p_hat = ...          # 2. forward pass: call the model on X                       (N, 1)
        loss = ...           # 3. cost: loss_fn(predictions, targets)                     tensor with one number
        history.append(...)  #    store the cost as a Python float (.item())
        ...                  # 4. backpropagation: compute the gradients of the cost
        ...                  # 5. update the parameters (optimizer)
    return history

In [ ]:
sprawdz_zadanie_7(train_torch, LogisticRegressionTorch)

Przy predykcji przełączamy model w tryb ewaluacji (`model.eval()`) i wyłączamy liczenie gradientów
(`torch.no_grad()`), co oszczędza czas i pamięć. 

Przygotowaliśmy poniżej funkcję `torch_proba`, która to robi i zamienia output PyTorcha
na NumPy, który przyjmuje nasz `plot_decision_boundary` i `plot_training`.

Trenujemy regresję logistyczną w PyTorchu na danych z Części I - wynik jest taki sam jak dla Twojej implementacji
w NumPy (krzywe różnią się tylko na starcie, bo modele zaczynają z innych losowych parametrów).

In [ ]:
def torch_proba(model):
    def predict_proba(X_np):
        model.eval()
        with torch.no_grad():
            return model(torch.tensor(X_np, dtype=torch.float32)).squeeze(1).numpy()
    return predict_proba


torch.manual_seed(0)
logreg_lin = LogisticRegressionTorch(2)
optimizer = optim.SGD(logreg_lin.parameters(), lr=0.5)
history_t = train_torch(logreg_lin, optimizer, X_lin_t, y_lin_t, epochs=100)
plot_training(history_t, torch_proba(logreg_lin), X_lin, y_lin, title="PyTorch", compare={"NumPy": history})

### Zbiór make_moons

[`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) generuje dwa
przeplatające się półkola. Trenujemy na nim tę samą regresję logistyczną (krzyżyki to błędne predykcje).

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=1000, noise=0.1, random_state=4)
X_moons_t = torch.tensor(X_moons, dtype=torch.float32)               # (N, 2)
y_moons_t = torch.tensor(y_moons, dtype=torch.float32).unsqueeze(1)  # (N, 1)

torch.manual_seed(0)
logreg_moons = LogisticRegressionTorch(2)
optimizer = optim.SGD(logreg_moons.parameters(), lr=0.5)
history_logreg = train_torch(logreg_moons, optimizer, X_moons_t, y_moons_t, epochs=1000)
plot_training(history_logreg, torch_proba(logreg_moons), X_moons, y_moons, title="Regresja logistyczna",
              mark_errors=True)

**Koszt przestaje maleć**, a granica decyzyjna regresji logistycznej to zawsze prosta - dłuższy trening ani
inna stała ucząca tego nie zmienią. Potrzebujemy modelu o większym capacity, który będzie w stanie nauczyć się nieliniowych zależności. 

Nieliniową granicę decyzyjną moglibyśmy uzyskać, definiując model wykorzystujący np. kombinacje kwadratowe cech (więcej o tym na PUMach). Zamiast ręcznie projektować takie cechy, wykorzystamy jednak sieć neuronową, która potrafi uczyć się złożonych, nieliniowych zależności bez konieczności jawnego definiowania ich przez nas.

---
## Część III - Sieć neuronowa MLP

Regresja logistyczna wyznacza liniową granicę decyzyjną w przestrzeni wejściowej, co ogranicza jej możliwości w przypadku bardziej złożonych zależności. Sieć neuronowa pozwala nam rozszerzyć przestrzeń rozwiązań - jest to możliwe dzięki warstwom ukrytym i nieliniowym funkcjom aktywacji, które przekształcają dane wejściowe w nowe reprezentacje, lepiej dopasowane do rozwiązywanego problemu.

Podstawowym elementem sieci neuronowej jest **neuron**: iloczyn skalarny wag z wejściem, bias i **funkcja
aktywacji (activation function)**. Nasza regresja logistyczna z Części I to dokładnie jeden neuron z aktywacją
sigmoidalną.

Odpowiednio skonstruowane sieci mogą pełnić funkcję uniwersalnych aproksymatorów. Oznacza to, że przy odpowiedniej liczbie neuronów i właściwie dobranych funkcjach aktywacji mogą z dowolną dokładnością przybliżać funkcje ciągłe na ograniczonych obszarach. W praktyce daje nam to dużą elastyczność w modelowaniu skomplikowanych zależności, choć samo istnienie takiej możliwości nie gwarantuje, że trening zawsze znajdzie dobre rozwiązanie.

W naszym zadaniu wykorzystamy architekturę nazwaną **MultiLayer Perceptron (MLP)** - sieć składającą się z kolejnych warstw neuronów.

**Warstwa ukryta (hidden layer)** to każda warstwa sieci leżąca między wejściem a wyjściem. Nazywamy ją „ukrytą”, bo jej wartości nie występują ani w danych wejściowych, ani w etykietach - nie mówimy sieci, co mają oznaczać. Sieć sama ustala je w trakcie treningu tak, żeby ułatwiały zadanie warstwie wyjściowej.

Nasza sieć ma **warstwę ukrytą** z 3 neuronami $\tanh$: każdy patrzy na wejście $x$ i zwraca jedną liczbę $h_j$. Wektor $h$ to
**ukryta reprezentacja (hidden representation)** - nowe cechy, których sieć uczy się sama. Warstwa wyjściowa to
zwykła regresja logistyczna, tyle że na cechach $h$ zamiast $x$.

<img src="figures/mlp_architektura.png" width="820" alt="architektura MLP">

Zapisana wzorami nasza sieć to dwie warstwy. Warstwa ukryta (macierz wag $W_1$ o kształcie $2 \times 3$):

$$
h = \tanh(x W_1 + b_1)
$$

i warstwa wyjściowa (macierz wag $W_2$ o kształcie $3 \times 1$):

$$
\hat p = \sigma(h W_2 + b_2)
$$

**Dlaczego aktywacja w warstwie ukrytej musi być nieliniowa?** Usuńmy $\tanh$ i zobaczmy, co zostanie z logitu
na wyjściu. Warstwa ukryta byłaby wtedy tylko przekształceniem liniowym:

$$
h = x W_1 + b_1
$$

Podstawiamy to do warstwy wyjściowej:

$$
z = h W_2 + b_2 = (x W_1 + b_1) W_2 + b_2
$$

i wymnażamy nawiasy:

$$
z = x \, (W_1 W_2) + (b_1 W_2 + b_2)
$$

Iloczyn $W_1 W_2$ to po prostu jedna macierz wag, a $b_1 W_2 + b_2$ - jeden wektor biasów. Dwie warstwy liniowe
dają więc znów **jedną** warstwę liniową, czyli zwykłą regresję logistyczną z granicą w postaci prostej - niezależnie
od liczby neuronów i warstw. Dopiero nieliniowa aktywacja pozwala sieci zginać granicę decyzyjną.

Warstwy łączymy
[`nn.Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html) (nie zwykłą listą
Pythona - jej parametry nie trafiłyby do `model.parameters()`, szczegóły w dokumentacji).

### Zadanie 8: Sieć MLP

Uzupełnij `MoonMLP`: w `self.layers` utwórz `nn.Sequential` z warstwami jak na schemacie, a w `forward` wywołaj go.
Przykład użycia `nn.Sequential`: [Build the Neural Network - nn.Sequential](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#nn-sequential).
Trenujemy Twoją funkcją `train_torch`, ale z optymalizatorem **Adam** (patrz opis optymalizatorów przy Zadaniu 7).

In [ ]:
class MoonMLP(nn.Module):
    # nn.Sequential example: https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#nn-sequential
    def __init__(self, hidden_size=3):
        super().__init__()
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        self.layers = nn.Sequential(
            ...,  # 1. nn.Linear: 2 -> hidden_size
            ...,  # 2. nn.Tanh
            ...,  # 3. nn.Linear: hidden_size -> 1
            ...,  # 4. nn.Sigmoid
        )

    def forward(self, X):
        # X: (N, 2) -> probabilities of class 1, shape (N, 1)
        # implement me! - delete the `raise` line and replace every `...` below
        raise NotImplementedError
        return ...  # pass X through self.layers

In [ ]:
sprawdz_zadanie_8(MoonMLP)

In [ ]:
torch.manual_seed(0)  # the network's cost is not convex - the result depends on the random start
mlp = MoonMLP(hidden_size=3)
optimizer = optim.Adam(mlp.parameters(), lr=0.01)
history_mlp = train_torch(mlp, optimizer, X_moons_t, y_moons_t, epochs=1000)

sprawdz_trening_mlp(mlp, X_moons_t, y_moons_t)
plot_training(history_mlp, torch_proba(mlp), X_moons, y_moons, title="MLP",
              compare={"regresja logistyczna": history_logreg})

Przez kilkaset epok koszt MLP stoi na poziomie regresji logistycznej (sieć zachowuje się jak model liniowy), a potem
gwałtownie spada, gdy sieć „odkrywa”, jak zgiąć granicę. Takie **plateau** są typowe przy uczeniu sieci.
Dokładność liczymy tu na zbiorze treningowym - w praktyce ocenia się model na osobnym zbiorze testowym.

Na plateau dobrze widać różnicę między optymalizatorami. Poniżej trenujemy tę samą sieć, z tego samego startu,
zwykłym SGD, SGD z momentum i Adamem.

In [ ]:
optimizers = {
    "SGD, lr = 0.1": lambda params: optim.SGD(params, lr=0.1),
    "SGD z momentum, lr = 0.1": lambda params: optim.SGD(params, lr=0.1, momentum=0.9),
    "Adam, lr = 0.01": lambda params: optim.Adam(params, lr=0.01),
}

plt.figure(figsize=(7, 3.8))
for name, make_optimizer in optimizers.items():
    torch.manual_seed(0)
    net = MoonMLP(hidden_size=3)
    hist = train_torch(net, make_optimizer(net.parameters()), X_moons_t, y_moons_t, epochs=1000)
    acc = np.mean((torch_proba(net)(X_moons) >= 0.5) == y_moons)
    plt.plot(hist, label=f"{name} - dokładność {acc:.1%}")
plt.xlabel("epoka")
plt.ylabel("koszt")
plt.title("Ta sama sieć, różne optymalizatory")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Zwykłe SGD utyka na plateau i przez 1000 epok nie wychodzi poza wynik regresji logistycznej. SGD z momentum najszybciej
schodzi na plateau, ale zaczyna z niego wychodzić dopiero pod koniec treningu. Adam wychodzi z plateau najwcześniej
i jako jedyny osiąga 100% dokładności.

Zajrzyjmy teraz do środka sieci. Po lewej proste, na których neurony ukryte zmieniają znak - z tych „cięć”
sieć składa granicę. Po prawej punkty w przestrzeni ukrytej $h$: tu klasy rozdziela już **płaszczyzna** ostatniej
warstwy.

In [ ]:
first_layer = mlp.layers[0]
W_h = first_layer.weight.detach().numpy()  # (3, 2)
b_h = first_layer.bias.detach().numpy()    # (3,)
with torch.no_grad():
    H = mlp.layers[1](first_layer(X_moons_t)).numpy()  # hidden representation, (N, 3)
W_out = mlp.layers[2].weight.detach().numpy().ravel()  # (3,)
b_out = mlp.layers[2].bias.item()

fig = plt.figure(figsize=(10, 4.2))
ax1 = fig.add_subplot(1, 2, 1)
plot_decision_boundary(torch_proba(mlp), X_moons, y_moons, ax=ax1, title="Proste neuronów ukrytych")
x1 = np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 50)
for j in range(len(b_h)):
    ax1.plot(x1, -(W_h[j, 0] * x1 + b_h[j]) / W_h[j, 1], "--", lw=2, label=f"neuron $h_{j + 1}$")
ax1.set_ylim(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5)
ax1.legend(loc="lower left", fontsize=8)

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(H[:, 0], H[:, 1], H[:, 2], c=y_moons, cmap="coolwarm", s=6, edgecolors="k", linewidths=0.2)
g1, g2 = np.meshgrid(np.linspace(-1, 1, 20), np.linspace(-1, 1, 20))
plane = -(W_out[0] * g1 + W_out[1] * g2 + b_out) / W_out[2]  # output layer boundary: W_out . h + b_out = 0
plane[np.abs(plane) > 1] = np.nan
ax2.plot_surface(g1, g2, plane, alpha=0.3, color="gray")
ax2.set_xlabel("$h_1$")
ax2.set_ylabel("$h_2$")
ax2.set_zlabel("$h_3$")
ax2.set_title("Przestrzeń ukryta")
ax2.view_init(elev=15, azim=-130)  # try other angles, e.g. azim=-60
fig.tight_layout()
plt.show()

Na koniec porównajmy kilka architektur: sieci z 1, 2 i 3 neuronami ukrytymi oraz sieć z 8 neuronami bez
nieliniowości (`nn.Identity()` zamiast `Tanh`).

In [ ]:
def build_and_train(model, epochs=1000):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    train_torch(model, optimizer, X_moons_t, y_moons_t, epochs=epochs)
    return model


torch.manual_seed(0)
models = {f"{h} neuron(y), tanh": build_and_train(MoonMLP(hidden_size=h)) for h in (1, 2, 3)}
models["8 neuronów, bez aktywacji"] = build_and_train(
    nn.Sequential(nn.Linear(2, 8), nn.Identity(), nn.Linear(8, 1), nn.Sigmoid()))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (name, net) in zip(axes, models.items()):
    acc = np.mean((torch_proba(net)(X_moons) >= 0.5) == y_moons)
    plot_decision_boundary(torch_proba(net), X_moons, y_moons, ax=ax, title=f"{name}: {acc:.1%}")
fig.tight_layout()
plt.show()

In [ ]:
podsumowanie()